# Classificacao via OpenRouter

Este notebook classifica os posts em `sentinel_replica_jsons_cleaned_eval/` usando o OpenRouter com o modelo `google/gemini-2.5-flash-preview-05-20` (ou outro model ID configurado em `MODEL`). Usa o cliente `AsyncOpenAI` apontado para `https://openrouter.ai/api/v1`.

In [1]:
from __future__ import annotations

import asyncio
import json
import os
from pathlib import Path
from typing import Any

import pandas as pd
from openai import AsyncOpenAI
from pydantic import BaseModel
from tqdm.asyncio import tqdm

INPUT_DIR = Path("../sentinel_replica_jsons_cleaned_eval")
OUTPUT_DIR = Path("../sentinel_replica_jsons_classified_eval")
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL = "google/gemini-3.1-flash-lite-preview"
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MAX_CONCURRENT = 16

JSON_FILES = sorted(INPUT_DIR.glob("*.json"))
len(JSON_FILES), JSON_FILES[:3]

(12,
 [PosixPath('../sentinel_replica_jsons_cleaned_eval/HackingBlogsGroup.json'),
  PosixPath('../sentinel_replica_jsons_cleaned_eval/PHOfficial.json'),
  PosixPath('../sentinel_replica_jsons_cleaned_eval/WokeIntelDrops.json')])

In [2]:
CONTENT_CLEANUP_LABELS = {
    "clean",
    "contains_markup",
    "contains_html",
    "contains_tracking_or_noise",
    "contains_forwarded_or_quoted_boilerplate",
    "contains_url_only_or_low_content",
    "unusable",
}

THREAT_LABELS = {
    "malware",
    "ransomware",
    "phishing",
    "credential_theft",
    "fraud_or_scam",
    "vulnerability_or_exploit",
    "data_breach_or_leak",
    "account_takeover",
    "ddos_or_disruption",
    "supply_chain_compromise",
    "insider_threat",
    "botnet_or_c2",
    "initial_access_activity",
    "reconnaissance",
    "privilege_escalation",
    "lateral_movement",
    "exfiltration",
    "wiper_or_destruction",
    "disinformation_or_influence",
    "physical_or_hybrid_threat",
    "other_cyber",
    "not_a_threat",
}

SEVERITY_LABELS = {"informational", "low", "medium", "high", "critical"}
CONFIDENCE_LABELS = {"low", "medium", "high"}
CONTENT_QUALITY_LABELS = {"high", "medium", "low", "unusable"}
TARGET_TYPE_LABELS = {"individual", "company", "government", "critical_infrastructure", "unknown"}
THREAT_ACTOR_TYPE_LABELS = {"criminal", "state", "hacktivist", "insider", "unknown"}
IOC_TYPE_LABELS = {"ip", "domain", "url", "hash", "email", "wallet", "handle"}

# Domains that are never a primary incident source
_SOCIAL_DOMAINS = {
    "linkedin.com", "lnkd.in", "twitter.com", "x.com", "t.me", "telegram.me",
    "youtube.com", "youtu.be", "facebook.com", "instagram.com", "tiktok.com",
    "github.com", "github.io", "udemy.com", "coursera.org", "udacity.com",
    "medium.com",
}


class ThreatClassification(BaseModel):
    cleaned_text: str
    content_cleanup: str
    needs_span_removal: bool
    needs_html_stripping: bool
    needs_noise_removal: bool
    content_quality: str
    language: str
    is_cyber_relevant: bool
    threat_label: str
    secondary_labels: list[str]
    severity: str
    confidence: str
    target_type: str
    threat_actor_type: str
    ioc_present: bool
    ioc_types: list[str]
    requires_human_review: bool
    rationale: str
    # --- Correlation fields (used to match against Hackmageddon ground truth) ---
    victim_name: str        # Canonical name of the attacked org/person; "" if not identifiable
    threat_actor_name: str  # Canonical threat actor or malware family name; "" if unknown
    primary_source_url: str # Main external news/report URL cited in the post; "" if none
    incident_date: str      # YYYY-MM-DD when the incident occurred; "" if not mentioned


PROMPT_SCHEMA_VIEW = {
    "cleaned_text": "string",
    "content_cleanup": sorted(CONTENT_CLEANUP_LABELS),
    "needs_span_removal": "boolean",
    "needs_html_stripping": "boolean",
    "needs_noise_removal": "boolean",
    "content_quality": sorted(CONTENT_QUALITY_LABELS),
    "language": "ISO-639-1 or unknown",
    "is_cyber_relevant": "boolean",
    "threat_label": sorted(THREAT_LABELS),
    "secondary_labels": sorted(THREAT_LABELS),
    "severity": sorted(SEVERITY_LABELS),
    "confidence": sorted(CONFIDENCE_LABELS),
    "target_type": sorted(TARGET_TYPE_LABELS),
    "threat_actor_type": sorted(THREAT_ACTOR_TYPE_LABELS),
    "ioc_present": "boolean",
    "ioc_types": sorted(IOC_TYPE_LABELS),
    "requires_human_review": "boolean",
    "rationale": "short string in pt-BR, max 240 chars",
    "victim_name": "official name of the attacked org/person, or empty string",
    "threat_actor_name": "canonical threat actor or malware family name, or empty string",
    "primary_source_url": "main news article/advisory URL cited in the post (exclude LinkedIn, Twitter/X, YouTube, Telegram, GitHub, courses), or empty string",
    "incident_date": "YYYY-MM-DD when the incident actually occurred (not the post date), or empty string",
}

In [3]:
import re as _re
from urllib.parse import urlsplit as _urlsplit

SYSTEM_PROMPT = """
Voce e um classificador de inteligencia de ameacas ciberneticas.

Retorne apenas um objeto JSON valido, sem markdown e sem texto extra.
Se a postagem nao for relevante para ameacas ciberneticas, use threat_label='not_a_threat'.
Preserve o idioma original em cleaned_text, apenas removendo ruido residual.
""".strip()


def build_prompt(post: dict[str, Any]) -> str:
    message = str(post.get("message", "")).strip()
    normalized = post.get("cleaning", {}).get("normalized_message", "")
    schema_json = json.dumps(PROMPT_SCHEMA_VIEW, ensure_ascii=False, indent=2)
    return f"""
Classifique a postagem abaixo seguindo exatamente o schema e os labels permitidos.

Regras gerais:
- Retorne um unico objeto JSON valido.
- secondary_labels deve ser uma lista sem duplicatas.
- ioc_types deve ser lista vazia se ioc_present=false.
- rationale deve ser curto, objetivo e em portugues.
- Use o campo normalized_message como apoio, mas baseie a classificacao no conteudo da postagem.

Regras para os campos de correlacao (victim_name, threat_actor_name, primary_source_url, incident_date):
- victim_name: nome oficial da organizacao ou pessoa atacada (ex: "Royal Mail", "LexisNexis").
  Use o nome canonico exato, sem descricoes genericas como "uma empresa europeia".
  Deixe vazio ("") se nao houver vitima especifica identificavel ou se threat_label='not_a_threat'.
- threat_actor_name: nome canonico do grupo, ator ou familia de malware responsavel (ex: "LockBit", "Killnet", "Cobalt Strike").
  Nao inclua sufixos de versao como "3.0" nem o tipo ("ransomware group"). Deixe vazio se desconhecido.
- primary_source_url: URL do artigo de noticias, relatorio ou advisory que a postagem esta referenciando ou citando.
  Exclua links de redes sociais (LinkedIn, Twitter/X, YouTube, Telegram), repositorios GitHub, cursos e plataformas educacionais.
  Se houver mais de uma URL de fonte, escolha a mais diretamente relacionada ao incidente. Deixe vazio se nao houver.
- incident_date: data em que o incidente ocorreu no formato YYYY-MM-DD. Nao use a data da postagem.
  Deixe vazio se a data do incidente nao for mencionada explicitamente.

Schema esperado:
{schema_json}

Postagem original:
{message}

Normalized message:
{normalized}
""".strip()


def _valid_source_url(url: str) -> str:
    url = url.strip()
    if not url.startswith("http"):
        return ""
    try:
        netloc = _urlsplit(url).netloc.lower().lstrip("www.")
    except ValueError:
        return ""
    if any(netloc == d or netloc.endswith("." + d) for d in _SOCIAL_DOMAINS):
        return ""
    return url


def _valid_incident_date(value: str) -> str:
    if _re.fullmatch(r"\d{4}-\d{2}-\d{2}", value.strip()):
        return value.strip()
    return ""


def normalize_classification(result: ThreatClassification) -> dict[str, Any]:
    data = result.model_dump()
    data["secondary_labels"] = list(dict.fromkeys(data.get("secondary_labels", [])))
    data["ioc_types"] = list(dict.fromkeys(data.get("ioc_types", [])))

    if data["content_cleanup"] not in CONTENT_CLEANUP_LABELS:
        data["content_cleanup"] = "unusable"
    if data["content_quality"] not in CONTENT_QUALITY_LABELS:
        data["content_quality"] = "unusable"
    if data["threat_label"] not in THREAT_LABELS:
        data["threat_label"] = "other_cyber"
    if data["severity"] not in SEVERITY_LABELS:
        data["severity"] = "informational"
    if data["confidence"] not in CONFIDENCE_LABELS:
        data["confidence"] = "low"
    if data["target_type"] not in TARGET_TYPE_LABELS:
        data["target_type"] = "unknown"
    if data["threat_actor_type"] not in THREAT_ACTOR_TYPE_LABELS:
        data["threat_actor_type"] = "unknown"

    data["secondary_labels"] = [label for label in data["secondary_labels"] if label in THREAT_LABELS]
    data["ioc_types"] = [ioc for ioc in data["ioc_types"] if ioc in IOC_TYPE_LABELS]

    if not data.get("ioc_present", False):
        data["ioc_types"] = []
    if data["threat_label"] == "not_a_threat":
        data["severity"] = "informational"
        data["victim_name"] = ""
        data["threat_actor_name"] = ""
        data["primary_source_url"] = ""
        data["incident_date"] = ""
    else:
        data["victim_name"] = data.get("victim_name", "").strip()
        data["threat_actor_name"] = data.get("threat_actor_name", "").strip()
        data["primary_source_url"] = _valid_source_url(data.get("primary_source_url", ""))
        data["incident_date"] = _valid_incident_date(data.get("incident_date", ""))

    return data


def fallback_classification(post: dict[str, Any], exc: Exception) -> dict[str, Any]:
    normalized = post.get("cleaning", {}).get("normalized_message", str(post.get("message", "")))
    return {
        "cleaned_text": normalized,
        "content_cleanup": "unusable",
        "needs_span_removal": False,
        "needs_html_stripping": False,
        "needs_noise_removal": False,
        "content_quality": "unusable",
        "language": "unknown",
        "is_cyber_relevant": False,
        "threat_label": "not_a_threat",
        "secondary_labels": [],
        "severity": "informational",
        "confidence": "low",
        "target_type": "unknown",
        "threat_actor_type": "unknown",
        "ioc_present": False,
        "ioc_types": [],
        "requires_human_review": True,
        "rationale": f"Falha ao classificar via OpenRouter: {type(exc).__name__}",
        "victim_name": "",
        "threat_actor_name": "",
        "primary_source_url": "",
        "incident_date": "",
    }

In [4]:
records = []
for json_file in JSON_FILES:
    with json_file.open("r", encoding="utf-8") as file:
        payload = json.load(file)
    if isinstance(payload, list):
        for item in payload:
            records.append({"source_file": json_file.name, **item})

df = pd.DataFrame(records)
df.head()

,source_file,date,message,id,cleaning,_id
0,HackingBlogsGroup.json,2025-06-19,🔥FREE NOTES API-HACKING BOOTCAMP DAY 1 : SETTI...,de9562f6-81dc-4df7-9250-d406b1b41a1a,{'normalized_message': 'free notes api-hacking...,NaN
1,HackingBlogsGroup.json,2025-06-09,🚨🚨A Secret Hacker GangExposed Is Exposing the ...,f80e9944-6e76-4d28-ba13-94752755e410,{'normalized_message': 'a secret hacker gangex...,NaN
2,HackingBlogsGroup.json,2025-06-08,🚨Delete These 20 Google Play Apps RIGHT NOW – ...,f77f1360-426c-496e-bc61-acd4fa01c6bb,{'normalized_message': 'delete these 20 google...,NaN
3,HackingBlogsGroup.json,2025-05-29,"🔥 364,000 Americans’ Data Exposed in LexisNexi...",7de299b3-81be-417c-a7e1-a0197d03db8a,{'normalized_message': '364 000 americans data...,NaN
4,HackingBlogsGroup.json,2025-05-24,⚠️ WARNING: TikTok Videos Offering Free Softwa...,67ac4da2-c4fb-4244-893f-e701d1d03f80,{'normalized_message': 'warning tiktok videos ...,NaN


In [5]:
_JSON_SCHEMA = {
    "name": "ThreatClassification",
    "schema": ThreatClassification.model_json_schema(),
    "strict": False,
}


def _load_existing_results() -> tuple[dict[str, list[dict[str, Any]]], set[tuple[str, str]]]:
    existing_by_file: dict[str, list[dict[str, Any]]] = {}
    seen_keys: set[tuple[str, str]] = set()

    for output_path in OUTPUT_DIR.glob("*.json"):
        try:
            with output_path.open("r", encoding="utf-8") as file:
                payload = json.load(file)
        except Exception:
            continue

        if not isinstance(payload, list):
            continue

        file_name = output_path.name
        existing_by_file[file_name] = payload

        for item in payload:
            if not isinstance(item, dict):
                continue
            post_id = item.get("id")
            if post_id is not None:
                seen_keys.add((file_name, str(post_id)))

    return existing_by_file, seen_keys


def _persist_results_for_file(file_name: str, items: list[dict[str, Any]]) -> None:
    output_path = OUTPUT_DIR / file_name
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(items, file, ensure_ascii=False, indent=2)
        file.write("\n")


def _record_key(post: dict[str, Any]) -> tuple[str, str] | None:
    source_file = post.get("source_file")
    post_id = post.get("id")
    if source_file is None or post_id is None:
        return None
    return str(source_file), str(post_id)


async def classify_post(
    client: AsyncOpenAI,
    semaphore: asyncio.Semaphore,
    post: dict[str, Any],
) -> dict[str, Any]:
    prompt = build_prompt(post)
    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "json_schema", "json_schema": _JSON_SCHEMA},
            )
            raw = response.choices[0].message.content or ""
            classification = normalize_classification(ThreatClassification.model_validate_json(raw))
        except Exception as exc:
            classification = fallback_classification(post, exc)

    enriched = dict(post)
    enriched["classification"] = classification
    return enriched


async def classify_all(records: list[dict[str, Any]], max_concurrent: int = MAX_CONCURRENT) -> list[dict[str, Any]]:
    write_lock = asyncio.Lock()
    semaphore = asyncio.Semaphore(max_concurrent)

    by_file, seen_keys = _load_existing_results()
    to_process = [p for p in records if (k := _record_key(p)) is None or k not in seen_keys]

    if not to_process:
        print("Nenhum novo post para classificar (todos ja processados no output).")
        return [item for items in by_file.values() for item in items]

    client = AsyncOpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)
    processed_count = 0

    async def process(post: dict[str, Any]) -> None:
        nonlocal processed_count
        classified = await classify_post(client, semaphore, post)
        async with write_lock:
            source_file = str(classified.get("source_file", "unknown.json"))
            by_file.setdefault(source_file, []).append(classified)
            _persist_results_for_file(source_file, by_file[source_file])
            key = _record_key(classified)
            if key is not None:
                seen_keys.add(key)
            processed_count += 1

    tasks = [asyncio.create_task(process(p)) for p in to_process]
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        await coro

    merged = [item for items in by_file.values() for item in items]
    print(f"Classificados agora: {processed_count} | Total no output: {len(merged)}")
    return merged

In [6]:
results = await classify_all(records)

by_file: dict[str, list[dict[str, Any]]] = {}
for item in results:
    by_file.setdefault(item["source_file"], []).append(item)

print(f"Done. {len(results)} posts disponiveis em {len(by_file)} arquivos no output.")

 11%|█▏        | 1138/9920 [02:34<19:49,  7.38it/s]


CancelledError: 